# 10 — Multimodal Analysis with CLIP, OpenCLIP & Remote Sensing Models

This notebook demonstrates how **multimodal embedding models** — originally designed to
align images and text in a shared vector space — can be repurposed for critical mineral
data science.

## The Core Idea

CLIP-family models learn a **joint embedding space** where semantically similar images and
text end up close together, regardless of modality.  We exploit this in two creative ways:

1. **Text-only embeddings** — encode rich mineral descriptions and compare how different
   model families represent geochemical and geopolitical concepts.
2. **Chart-as-image embeddings** — render production time-series as matplotlib figures,
   then feed those images to CLIP's vision encoder.  This lets us find countries whose
   production *curves look visually similar*, enabling pattern search without writing a
   single feature-engineering function.

A third section demonstrates how **RemoteCLIP** and **GeoRSCLIP** — CLIP variants
fine-tuned on remote sensing imagery — would be used to classify satellite images of
mining sites using only natural language prompts.

## Models Featured

| Model | Backbone | Training Data | Best For |
|---|---|---|---|
| **OpenCLIP** (ViT-B/32) | Vision Transformer | LAION-2B (2 billion image-text pairs) | General image + text alignment |
| **SigLIP** (ViT-B/16) | Vision Transformer | Google's WebLI dataset | Improved zero-shot classification via sigmoid loss |
| **RemoteCLIP** | ViT-L/14 | Remote sensing datasets (RS5M) | Satellite image + text matching |
| **GeoRSCLIP** | ViT-H/14 | 5M geo remote sensing pairs | Geoscience image retrieval |

> **Note:** RemoteCLIP and GeoRSCLIP require satellite imagery to unlock their full
> potential.  In this notebook we demonstrate the text-embedding side and show how
> cross-modal inference would work once satellite images are available.

## 0. Setup

Install dependencies and configure globals.  `open_clip_torch` bundles pretrained OpenCLIP
weights; `transformers` provides SigLIP via the HuggingFace hub.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pathlib
import io

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
from PIL import Image
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — required for rendering to buffers
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = pathlib.Path("../data/bgs_data")
CSV_PATH = DATA_DIR / "bgs_critical_minerals_production.csv"

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEMPLATE = "plotly_white"

print(f"CSV path : {CSV_PATH}")
print(f"Exists   : {CSV_PATH.exists()}")
print(f"Device   : {DEVICE}")

## 1. Load and Prepare BGS Production Data

We filter to `Production` records, coerce numeric columns, and keep only rows with valid
quantities.  The resulting `df` is reused throughout every section of this notebook.

In [ ]:
raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Raw rows : {len(raw):,}")
print(f"Columns  : {list(raw.columns)}")

# Keep Production records only
df = raw[raw["statistic_type"].str.strip().str.lower() == "production"].copy()
print(f"After production filter: {len(df):,} rows")

# Coerce numerics
df["year"]     = pd.to_numeric(df["year"],     errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df = df.dropna(subset=["year", "quantity", "country", "commodity"])
df = df[df["quantity"] > 0].copy()
df["year"] = df["year"].astype(int)

# Normalise commodity names
df["commodity"] = df["commodity"].str.strip().str.lower()

print(f"After cleaning          : {len(df):,} rows")
print(f"Year range              : {df['year'].min()} — {df['year'].max()}")
print(f"Unique commodities      : {df['commodity'].nunique()}")
print(f"Unique countries        : {df['country'].nunique()}")

df[["commodity", "country", "year", "quantity", "units"]].head()

In [ ]:
# Quick overview: total production by commodity (last 5 years)
max_year  = df["year"].max()
recent_df = df[df["year"] >= max_year - 4]

top_commodities = (
    recent_df.groupby("commodity")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

fig = px.bar(
    top_commodities,
    x="quantity", y="commodity",
    orientation="h",
    title=f"Top 20 Commodities by Total Production ({max_year-4}–{max_year})",
    labels={"quantity": "Total Quantity", "commodity": "Commodity"},
    template=TEMPLATE, height=550,
)
fig.show()

## 2. OpenCLIP — Text Embeddings for Critical Minerals

**OpenCLIP** is the open-source reimplementation of OpenAI's CLIP, trained by LAION on
2 billion image-text pairs from the web.  We use the `ViT-B/32` checkpoint pretrained on
`laion2b_s34b_b79k` (34 billion samples seen, 79K batch size).

Here we feed **text-only** descriptions into the text encoder to obtain 512-dimensional
embeddings for each commodity.  These can then be compared via cosine similarity to
cluster minerals by their semantic proximity in CLIP's representation space.

In [ ]:
import open_clip

# Load OpenCLIP ViT-B/32 trained on LAION-2B
print("Loading OpenCLIP ViT-B/32 (laion2b_s34b_b79k) ...")
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k'
)
tokenizer = open_clip.get_tokenizer('ViT-B-32')
clip_model = clip_model.to(DEVICE).eval()
print("Model loaded.")

In [ ]:
# Build one description per commodity
commodity_list = sorted(df["commodity"].unique())
print(f"Total commodities: {len(commodity_list)}")

mineral_descriptions = []
for commodity in commodity_list:
    subset = df[df["commodity"] == commodity]
    n_countries  = subset["country"].nunique()
    year_span    = f"{int(subset['year'].min())}–{int(subset['year'].max())}"
    latest_year  = int(subset["year"].max())
    latest       = subset[subset["year"] == latest_year]
    top_row      = latest.nlargest(1, "quantity")
    top_country  = top_row["country"].iloc[0] if not top_row.empty else "unknown"

    desc = (
        f"Mining and production of {commodity}, a critical mineral used in "
        f"industrial and technological applications, produced across "
        f"{n_countries} countries from {year_span}. "
        f"Largest producer in {latest_year}: {top_country}."
    )
    mineral_descriptions.append({"commodity": commodity, "description": desc})

desc_df = pd.DataFrame(mineral_descriptions)
print(desc_df[["commodity", "description"]].head(4).to_string(index=False))

In [ ]:
# Encode with CLIP text encoder
# CLIP tokenizer truncates at 77 tokens — descriptions are short enough to fit.
with torch.no_grad():
    tokens         = tokenizer(desc_df["description"].tolist()).to(DEVICE)
    text_features  = clip_model.encode_text(tokens)
    text_features  = text_features / text_features.norm(dim=-1, keepdim=True)
    clip_text_emb  = text_features.cpu().numpy()

print(f"OpenCLIP text embeddings shape: {clip_text_emb.shape}")
# Expected: (n_commodities, 512)

In [ ]:
# Cosine similarity heatmap — OpenCLIP text embeddings
clip_sim = cosine_similarity(clip_text_emb)

# Show only first 25 commodities for readability
N_SHOW    = min(25, len(commodity_list))
labels_25 = [c[:30] for c in commodity_list[:N_SHOW]]

fig = px.imshow(
    clip_sim[:N_SHOW, :N_SHOW],
    x=labels_25, y=labels_25,
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    title="OpenCLIP Text Embedding Cosine Similarity — Critical Minerals",
    labels=dict(color="Cosine Sim"),
    template=TEMPLATE,
)
fig.update_layout(height=650, width=700,
                  xaxis_tickangle=-45)
fig.show()

## 3. SigLIP — Google's Improved CLIP

**SigLIP** (Sigmoid Loss for Language-Image Pre-training) replaces CLIP's contrastive
softmax loss with a **per-pair sigmoid loss**.  This removes the need for a global
normalisation step across the batch and allows training on much larger batch sizes,
consistently improving zero-shot classification accuracy.

Key differences from standard CLIP:
- Sigmoid loss instead of softmax — each image-text pair is scored independently.
- No hard negatives required — scales to arbitrarily large batches.
- Stronger zero-shot transfer on fine-grained classification benchmarks.

We load `google/siglip-base-patch16-224` from HuggingFace and embed the same mineral
descriptions to compare how the two model families structure the embedding space.

In [ ]:
from transformers import AutoTokenizer, AutoModel

print("Loading SigLIP (google/siglip-base-patch16-224) ...")
siglip_tokenizer = AutoTokenizer.from_pretrained("google/siglip-base-patch16-224")
siglip_model     = AutoModel.from_pretrained("google/siglip-base-patch16-224").to(DEVICE).eval()
print("Model loaded.")

In [ ]:
# Encode with SigLIP text encoder — process in small batches to avoid OOM on CPU
BATCH_SIZE = 16
descriptions = desc_df["description"].tolist()
all_siglip_embs = []

with torch.no_grad():
    for i in range(0, len(descriptions), BATCH_SIZE):
        batch = descriptions[i : i + BATCH_SIZE]
        inputs = siglip_tokenizer(
            batch, padding="max_length", truncation=True,
            max_length=64, return_tensors="pt"
        ).to(DEVICE)
        emb = siglip_model.get_text_features(**inputs)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        all_siglip_embs.append(emb.cpu())

siglip_text_emb = torch.cat(all_siglip_embs, dim=0).numpy()
print(f"SigLIP text embeddings shape: {siglip_text_emb.shape}")
# Expected: (n_commodities, 768)

In [ ]:
# Side-by-side comparison: OpenCLIP vs SigLIP top-5 similar commodities
siglip_sim = cosine_similarity(siglip_text_emb)

QUERY_COMMODITY = "lithium minerals"
if QUERY_COMMODITY not in commodity_list:
    QUERY_COMMODITY = commodity_list[0]

q_idx = commodity_list.index(QUERY_COMMODITY)

# Mask self-similarity
clip_sims_q   = clip_sim[q_idx].copy();   clip_sims_q[q_idx]   = -99
siglip_sims_q = siglip_sim[q_idx].copy(); siglip_sims_q[q_idx] = -99

clip_top5   = np.argsort(clip_sims_q)[::-1][:5]
siglip_top5 = np.argsort(siglip_sims_q)[::-1][:5]

print(f"Query commodity: '{QUERY_COMMODITY}'")
print("\nOpenCLIP top-5 similar:")
for idx in clip_top5:
    print(f"  [{clip_sims_q[idx]:.4f}]  {commodity_list[idx]}")

print("\nSigLIP top-5 similar:")
for idx in siglip_top5:
    print(f"  [{siglip_sims_q[idx]:.4f}]  {commodity_list[idx]}")

In [ ]:
# Scatter: CLIP similarity vs SigLIP similarity — model agreement / disagreement
compare_df = pd.DataFrame({
    "commodity"  : commodity_list,
    "clip_sim"   : clip_sim[q_idx],
    "siglip_sim" : siglip_sim[q_idx],
})
compare_df = compare_df[compare_df["commodity"] != QUERY_COMMODITY]

fig = px.scatter(
    compare_df,
    x="clip_sim", y="siglip_sim",
    hover_name="commodity",
    title=f"OpenCLIP vs SigLIP Similarity to '{QUERY_COMMODITY}'",
    labels={
        "clip_sim"  : "OpenCLIP cosine similarity",
        "siglip_sim": "SigLIP cosine similarity",
    },
    template=TEMPLATE,
    height=500, width=650,
)
lo = min(compare_df[["clip_sim", "siglip_sim"]].min())
hi = max(compare_df[["clip_sim", "siglip_sim"]].max())
fig.add_trace(go.Scatter(
    x=[lo, hi], y=[lo, hi],
    mode="lines", name="y = x (perfect agreement)",
    line=dict(color="grey", dash="dash"),
))
fig.show()

## 4. Visual Pattern Matching — Chart-as-Image Embeddings

This is the most creative section.  The idea:

1. For each (mineral, country) pair, render the production time-series as a **matplotlib
   figure** saved to an in-memory PNG buffer.
2. Open the PNG as a PIL `Image` and pass it through OpenCLIP's **vision encoder**.
3. The resulting 512-D embedding encodes the *visual shape* of the production curve —
   not the raw numbers, but the visual gestalt (slope, volatility, kinks).
4. Use cosine similarity in this image-embedding space to find countries whose production
   histories **look alike** — a form of visual similarity search.

This approach bypasses manual feature engineering (no hand-coded slope or variance features)
and naturally captures complex curve shapes that are hard to describe analytically.

In [ ]:
def render_timeseries_image(
    years,
    quantities,
    title: str = "",
    size: tuple = (224, 224),
) -> Image.Image:
    """Render a production time-series as a PIL Image suitable for CLIP."""
    fig, ax = plt.subplots(figsize=(3, 2), dpi=75)
    ax.plot(years, quantities, 'b-', linewidth=2)
    ax.fill_between(years, quantities, alpha=0.25, color='steelblue')
    ax.set_title(title, fontsize=7, pad=3)
    ax.tick_params(labelsize=5)
    ax.set_xlabel("Year", fontsize=6)
    ax.set_ylabel("Qty",  fontsize=6)
    plt.tight_layout(pad=0.5)

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=75, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)

    img = Image.open(buf).convert('RGB').resize(size, Image.LANCZOS)
    return img


# Generate chart images for key minerals
KEY_MINERALS = [
    "lithium minerals",
    "cobalt, mine",
    "nickel, mine",
    "graphite",
    "rare earth minerals",
    "copper, mine",
    "manganese ore",
    "phosphate rock",
]

available_minerals = [m for m in KEY_MINERALS if m in df["commodity"].unique()]
print(f"Key minerals found in data: {available_minerals}")

# Fall back to top-N commodities if none of the key minerals match exactly
if not available_minerals:
    available_minerals = (
        df.groupby("commodity")["quantity"].sum()
        .sort_values(ascending=False)
        .head(8).index.tolist()
    )
    print(f"Using fallback minerals: {available_minerals}")

chart_images   = []
chart_metadata = []
MIN_YEARS      = 5   # require at least 5 data points per series

for mineral in available_minerals:
    mineral_df    = df[df["commodity"] == mineral]
    top_countries = (
        mineral_df.groupby("country")["quantity"]
        .sum().nlargest(8).index.tolist()
    )

    for country in top_countries:
        series = (
            mineral_df[mineral_df["country"] == country]
            .groupby("year")["quantity"].sum()
            .reset_index()
            .sort_values("year")
        )
        if len(series) < MIN_YEARS:
            continue

        short_title = f"{mineral[:20]} / {country[:15]}"
        img = render_timeseries_image(
            series["year"].values,
            series["quantity"].values,
            title=short_title,
        )
        chart_images.append(img)
        chart_metadata.append({
            "commodity": mineral,
            "country"  : country,
            "n_years"  : len(series),
        })

print(f"Generated {len(chart_images)} chart images")

In [ ]:
# Encode chart images with CLIP vision encoder
# preprocess applies the standard CLIP normalisation (resize, center-crop, normalize).
print("Encoding chart images with OpenCLIP vision encoder ...")

IMG_BATCH = 32
all_img_embs = []

with torch.no_grad():
    for i in range(0, len(chart_images), IMG_BATCH):
        batch_imgs   = chart_images[i : i + IMG_BATCH]
        img_tensors  = torch.stack([preprocess(img) for img in batch_imgs]).to(DEVICE)
        img_features = clip_model.encode_image(img_tensors)
        img_features = img_features / img_features.norm(dim=-1, keepdim=True)
        all_img_embs.append(img_features.cpu())

chart_embeddings = torch.cat(all_img_embs, dim=0).numpy()
print(f"Chart embeddings shape: {chart_embeddings.shape}")
# Expected: (n_charts, 512)

In [ ]:
# Visual similarity search — find production curves that look alike
QUERY_IDX  = 0   # Change this index to explore different query charts
query_meta = chart_metadata[QUERY_IDX]
print(f"Query chart : {query_meta['commodity']} — {query_meta['country']}")
print(f"Data points : {query_meta['n_years']} years\n")

sims = cosine_similarity(
    chart_embeddings[QUERY_IDX : QUERY_IDX + 1],
    chart_embeddings
)[0]

top_indices = np.argsort(sims)[::-1]
top_indices = [i for i in top_indices if i != QUERY_IDX][:5]

print("Most visually similar production curves (by CLIP image embedding):")
for rank, idx in enumerate(top_indices, start=1):
    meta = chart_metadata[idx]
    print(f"  {rank}. [{sims[idx]:.4f}]  {meta['commodity']} — {meta['country']}")

In [ ]:
# Display query chart + top-3 matches side by side
display_indices = [QUERY_IDX] + top_indices[:3]
titles          = ["QUERY"] + [f"#{r+1} [{sims[i]:.3f}]" for r, i in enumerate(top_indices[:3])]

fig_grid, axes = plt.subplots(1, len(display_indices), figsize=(14, 3.5))

for ax, idx, title_prefix in zip(axes, display_indices, titles):
    meta = chart_metadata[idx]
    ax.imshow(chart_images[idx])
    ax.set_title(
        f"{title_prefix}\n{meta['commodity'][:22]}\n{meta['country'][:18]}",
        fontsize=8, pad=4
    )
    ax.axis('off')

fig_grid.suptitle(
    "CLIP Vision-Embedding Visual Similarity Search — Production Curve Shapes",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig("clip_similarity_grid.png", dpi=100, bbox_inches='tight')
plt.show()
print("Grid saved to clip_similarity_grid.png")

## 5. Text-to-Chart Cross-Modal Search

CLIP's shared embedding space means we can describe a *visual pattern* in natural language
and retrieve the matching chart images — even though the model was never explicitly trained
on production charts.

This is **zero-shot cross-modal retrieval**: the text encoder and vision encoder project
into the same space, so a text embedding of "exponential growth" should be close to
images of exponentially growing curves.

We test five pattern descriptions and retrieve the top-matching chart images.

In [ ]:
# Text pattern queries describing visual curve shapes
pattern_queries = [
    "steadily increasing production growth over time",
    "sharp decline and crash in production",
    "stable and flat production with no change",
    "exponential growth in mineral production",
    "volatile and unstable production with many fluctuations",
]

with torch.no_grad():
    query_tokens   = tokenizer(pattern_queries).to(DEVICE)
    query_features = clip_model.encode_text(query_tokens)
    query_features = query_features / query_features.norm(dim=-1, keepdim=True)
    query_embs     = query_features.cpu().numpy()

print(f"Pattern query embeddings shape: {query_embs.shape}")

# Cross-modal search: text query -> chart images
results = []
for i, query in enumerate(pattern_queries):
    sims    = cosine_similarity(query_embs[i : i + 1], chart_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:3]
    for rank, idx in enumerate(top_idx, start=1):
        meta = chart_metadata[idx]
        results.append({
            "query"     : query,
            "rank"      : rank,
            "similarity": round(float(sims[idx]), 4),
            "commodity" : meta["commodity"],
            "country"   : meta["country"],
        })

results_df = pd.DataFrame(results)

for query in pattern_queries:
    sub = results_df[results_df["query"] == query]
    print(f"\nQuery: '{query}'")
    for _, row in sub.iterrows():
        print(f"  #{row['rank']} [{row['similarity']:.4f}]  "
              f"{row['commodity']} — {row['country']}")

In [ ]:
# Boxplot: cross-modal similarity distribution per query
fig = go.Figure()

for i, query in enumerate(pattern_queries):
    sims = cosine_similarity(query_embs[i : i + 1], chart_embeddings)[0]
    fig.add_trace(go.Box(
        y=sims,
        name=query[:45] + ("..." if len(query) > 45 else ""),
        boxpoints="outliers",
        marker_size=4,
    ))

fig.update_layout(
    title="Cross-Modal Similarity Distribution: Text Query to Chart Images",
    yaxis_title="Cosine Similarity",
    xaxis_title="Text Query",
    template=TEMPLATE,
    height=500,
    xaxis_tickangle=-25,
)
fig.show()

## 6. Remote Sensing CLIP for Mining Site Classification

**RemoteCLIP** and **GeoRSCLIP** are CLIP variants fine-tuned specifically on remote sensing
(satellite) imagery.  They dramatically outperform vanilla CLIP on tasks like:

- Classifying land cover from satellite images (open pit mine vs. forest vs. urban area)
- Retrieving satellite images matching a text description
- Detecting infrastructure (tailings ponds, processing plants, haul roads)

### How It Would Work in Practice

```
Satellite image (224x224 px)
       |
       v
RemoteCLIP Vision Encoder --> image embedding (768-D)
                                      |
                                      v
                               cosine_similarity
                                      |
                                      v
Text: "open pit mine" --> RemoteCLIP Text Encoder --> text embedding (768-D)
```

The label with the highest cosine similarity wins — **zero-shot classification** with no
task-specific training data required.

### In This Notebook

Since we do not have satellite imagery in the repository, we demonstrate the **text-embedding
side**: how different mining-related prompts cluster in CLIP space, and which prompts are
semantically close vs. far apart.  This analysis directly informs which label sets to use
when deploying a real satellite classifier.

In [ ]:
# Mining-site classification prompts
# These are the label strings you would use with RemoteCLIP / GeoRSCLIP
# to zero-shot classify satellite images of mining regions.
mining_prompts = [
    # Mining infrastructure
    "an aerial view of an open pit mine",
    "a satellite image of an underground mine entrance",
    "tailings pond at a mining site",
    "mineral processing plant with conveyor belts",
    "mineral exploration drilling site in a remote area",
    "haul road through an active mining operation",
    # Environmental context
    "pristine forest with no mining activity",
    "deforested area near a mining operation",
    "river with sedimentation from mine runoff",
    # Land use
    "agricultural land near a mining operation",
    "urban industrial area with warehouses",
    "remote arid landscape with geological formations",
]

print(f"Mining prompts: {len(mining_prompts)}")

# Encode with OpenCLIP text encoder
# (RemoteCLIP uses the same tokenizer — swapping model weights is the only change)
with torch.no_grad():
    mining_tokens   = tokenizer(mining_prompts).to(DEVICE)
    mining_features = clip_model.encode_text(mining_tokens)
    mining_features = mining_features / mining_features.norm(dim=-1, keepdim=True)
    mining_embs     = mining_features.cpu().numpy()

print(f"Mining prompt embeddings shape: {mining_embs.shape}")

In [ ]:
# Cosine similarity heatmap — mining prompts
mining_sim    = cosine_similarity(mining_embs)
prompt_labels = [p[:45] + ("..." if len(p) > 45 else "") for p in mining_prompts]

fig = px.imshow(
    mining_sim,
    x=prompt_labels, y=prompt_labels,
    color_continuous_scale="Blues",
    zmin=0, zmax=1,
    title="CLIP Text Embedding Similarity: Mining-Related Prompts (RemoteCLIP label candidates)",
    labels=dict(color="Cosine Sim"),
    template=TEMPLATE,
)
fig.update_layout(
    height=650, width=750,
    xaxis_tickangle=-50,
    xaxis_tickfont_size=9,
    yaxis_tickfont_size=9,
)
fig.show()

In [ ]:
# Simulated zero-shot classification demo
# In a real deployment, replace synthetic_image_embs with actual RemoteCLIP
# vision encoder outputs from satellite image tiles.
# Here we simulate N images whose embeddings are noisy versions of the label
# embeddings to illustrate the classification pipeline end-to-end.

np.random.seed(42)
N_SIM_IMAGES   = 20
TRUE_LABEL_IDX = np.random.randint(0, len(mining_prompts), size=N_SIM_IMAGES)

# Add Gaussian noise to simulate imperfect image embeddings
NOISE_STD = 0.25
synthetic_image_embs = (
    mining_embs[TRUE_LABEL_IDX]
    + np.random.normal(0, NOISE_STD, (N_SIM_IMAGES, mining_embs.shape[1]))
)
# L2-normalise
norms = np.linalg.norm(synthetic_image_embs, axis=1, keepdims=True)
synthetic_image_embs = synthetic_image_embs / norms

# Zero-shot classification: argmax cosine similarity against all label embeddings
logit_matrix  = cosine_similarity(synthetic_image_embs, mining_embs)
predicted_idx = np.argmax(logit_matrix, axis=1)

accuracy = (predicted_idx == TRUE_LABEL_IDX).mean()
print(f"Simulated zero-shot accuracy: {accuracy:.0%} "
      f"({int(accuracy * N_SIM_IMAGES)}/{N_SIM_IMAGES} correct)")
print()
print(f"{'Image':>5}  {'True Label':<50}  {'Predicted':<50}  Correct")
print("-" * 120)
for i in range(N_SIM_IMAGES):
    true_lbl = mining_prompts[TRUE_LABEL_IDX[i]][:47]
    pred_lbl = mining_prompts[predicted_idx[i]][:47]
    correct  = "YES" if predicted_idx[i] == TRUE_LABEL_IDX[i] else "NO "
    print(f"{i+1:>5}  {true_lbl:<50}  {pred_lbl:<50}  {correct}")

In [ ]:
# Confusion heatmap for simulated classification
from sklearn.metrics import confusion_matrix

cm           = confusion_matrix(TRUE_LABEL_IDX, predicted_idx,
                                labels=list(range(len(mining_prompts))))
short_labels = [p[:35] + ("..." if len(p) > 35 else "") for p in mining_prompts]

fig = px.imshow(
    cm,
    x=short_labels, y=short_labels,
    color_continuous_scale="Greens",
    title="Simulated Zero-Shot Classification Confusion Matrix (RemoteCLIP pipeline demo)",
    labels=dict(x="Predicted", y="True", color="Count"),
    template=TEMPLATE,
)
fig.update_layout(
    height=600, width=750,
    xaxis_tickangle=-50,
    xaxis_tickfont_size=8,
    yaxis_tickfont_size=8,
)
fig.show()

## 7. Embedding Space Visualisation — UMAP Projection

We combine the commodity text embeddings from both models and project them into 2-D with
UMAP.  Points from the same commodity encoded by different models reveal how aligned
(or divergent) OpenCLIP and SigLIP representations are in practice.

In [ ]:
import umap as umap_lib

# PCA-reduce SigLIP to 512-D to match OpenCLIP dimensionality before stacking
pca       = PCA(n_components=min(512, siglip_text_emb.shape[1]), random_state=42)
siglip_512 = pca.fit_transform(siglip_text_emb)
siglip_512 = siglip_512 / (np.linalg.norm(siglip_512, axis=1, keepdims=True) + 1e-9)

combined_embs   = np.vstack([clip_text_emb, siglip_512])
combined_labels = ["OpenCLIP"] * len(clip_text_emb) + ["SigLIP"] * len(siglip_512)
combined_names  = commodity_list + commodity_list

print(f"Combined embedding matrix: {combined_embs.shape}")

reducer = umap_lib.UMAP(
    n_components=2,
    n_neighbors=10,
    min_dist=0.2,
    metric="cosine",
    random_state=42,
)
umap_2d = reducer.fit_transform(combined_embs)
print(f"UMAP output shape: {umap_2d.shape}")

In [ ]:
# Interactive UMAP scatter — OpenCLIP vs SigLIP embedding alignment
umap_df = pd.DataFrame({
    "umap_x"   : umap_2d[:, 0],
    "umap_y"   : umap_2d[:, 1],
    "model"    : combined_labels,
    "commodity": combined_names,
})

fig = px.scatter(
    umap_df,
    x="umap_x", y="umap_y",
    color="model",
    hover_name="commodity",
    symbol="model",
    color_discrete_map={"OpenCLIP": "royalblue", "SigLIP": "crimson"},
    title="UMAP Projection: OpenCLIP vs SigLIP — Critical Mineral Text Embeddings",
    labels={
        "umap_x": "UMAP Dimension 1",
        "umap_y": "UMAP Dimension 2",
        "model" : "Model",
    },
    template=TEMPLATE,
    height=600, width=850,
    opacity=0.8,
)
fig.update_traces(marker_size=8)
fig.update_layout(
    legend=dict(title="Model", orientation="v"),
    font=dict(size=12),
)
fig.show()

## 8. Summary & Next Steps

### What Each Model Offers

| Model | Strengths | Limitations for Mineral Analysis |
|---|---|---|
| **OpenCLIP ViT-B/32** | Fast, well-tested, large community | General web imagery — not tuned to geology or satellite data |
| **SigLIP ViT-B/16** | Better zero-shot accuracy, sigmoid loss scales well | Same domain gap as OpenCLIP |
| **RemoteCLIP** | Trained on satellite imagery, understands land cover | Requires real satellite images to unlock vision encoder |
| **GeoRSCLIP** | 5M geo-paired training samples, ViT-H backbone | Larger model — slower inference; same image requirement |

### Limitations of Text-Only Analysis

- The descriptions we built are **template-generated** and relatively uniform — a more
  powerful approach would use LLM-generated rich descriptions that capture geology,
  economics, and geopolitics.
- CLIP text encoders are **77-token limited** (standard CLIP tokenizer); longer context
  models like LongCLIP or ALIGN handle richer descriptions.
- Text-to-image cross-modal matching works best when the model was trained
  on domain-similar content.  Vanilla OpenCLIP has limited exposure to scientific charts.

### Recommended Next Steps

1. **Integrate satellite imagery** — pair mining-site coordinates from the BGS data with
   Sentinel-2 or Landsat tiles (freely available via Google Earth Engine or AWS Open Data)
   and run RemoteCLIP's vision encoder on each tile.
2. **Fine-tune on mineral charts** — a small fine-tuning run of CLIP on (chart image,
   description) pairs from this dataset would dramatically improve cross-modal retrieval.
3. **LongCLIP or ALIGN** — explore models that accept longer text inputs for richer
   geochemical descriptions.
4. **Production change detection** — use RemoteCLIP to flag satellite tiles whose
   embeddings shift significantly over time, potentially detecting new mine openings or
   closures.

### References

- Radford et al. (2021). [Learning Transferable Visual Models From Natural Language Supervision](https://arxiv.org/abs/2103.00020) — Original CLIP paper.
- Cherti et al. (2022). [Reproducible Scaling Laws for Contrastive Language-Image Learning](https://arxiv.org/abs/2212.07143) — OpenCLIP.
- Zhai et al. (2023). [Sigmoid Loss for Language Image Pre-Training](https://arxiv.org/abs/2303.15343) — SigLIP.
- Chen et al. (2023). [RemoteCLIP: A Vision Language Foundation Model for Remote Sensing](https://arxiv.org/abs/2306.11029).
- Zhang et al. (2024). [GeoRSCLIP: A Universal Image-Text Retrieval Model for Remote Sensing](https://arxiv.org/abs/2404.09260).